# Portfolio Hybride IBKR + Binance — Phase 1 Research (Crypto Sleeve)

**Phase 1** : Binance crypto sleeve analysis (equity sleeve deferred to Phase 2 - Security Master subscription required).

**Sleeve Binance (100% this phase)** :
- EMA-Cross-Crypto (BTC): 50%
- MultiCanal (equal-weight crypto basket): 30%
- HAR-RV-J vol-target BTC (M12): 20%

**Objectifs** :
1. Charger les donnees historiques crypto Binance
2. Simuler les signaux de chaque sous-strategie crypto
3. Calculer la matrice de correlations inter-strategies
4. Estimer le Sharpe blend theorique du sleeve crypto
5. Analyse sensibilite allocation intra-crypto

In [1]:
from AlgorithmImports import *
import numpy as np
import pandas as pd
from itertools import combinations

qb = QuantBook()

# Binance sleeve assets (Phase 1: crypto-only, equity sleeve deferred to Phase 2)
crypto_symbols = {}
for t in ['BTCUSDT', 'ETHUSDT', 'SOLUSDT', 'ADAUSDT', 'LTCUSDT', 'XRPUSDT']:
    crypto_symbols[t] = qb.add_crypto(t, Resolution.DAILY, Market.Binance).symbol

print(f'Binance assets loaded: {len(crypto_symbols)}')
for name, sym in crypto_symbols.items():
    print(f'  {name}: {sym}')

Binance assets loaded: 6
  BTCUSDT: BTCUSDT
  ETHUSDT: ETHUSDT
  SOLUSDT: SOLUSDT
  ADAUSDT: ADAUSDT
  LTCUSDT: LTCUSDT
  XRPUSDT: XRPUSDT


In [2]:
# Load crypto historical data
start = datetime(2019, 1, 1)
end = datetime(2025, 12, 31)

crypto_history = qb.history(list(crypto_symbols.values()), start, end, Resolution.DAILY)
crypto_close = crypto_history['close'].unstack(level=0)
crypto_returns = crypto_close.pct_change().dropna()
print(f'Crypto data: {crypto_returns.shape[0]} days, {crypto_returns.shape[1]} assets')
print(f'Date range: {crypto_returns.index[0]} to {crypto_returns.index[-1]}')
print()
print(crypto_close.tail())

Crypto data: 1967 days, 6 assets
Date range: 2020-08-13 00:00:00 to 2025-12-31 00:00:00

symbol      ADAUSDT   BTCUSDT  ETHUSDT  LTCUSDT  SOLUSDT  XRPUSDT
time                                                             
2025-12-27   0.3496  87369.56  2927.99    76.87   122.26   1.8437
2025-12-28   0.3700  87877.01  2949.05    80.33   124.73   1.8735
2025-12-29   0.3688  87952.71  2950.91    78.73   125.27   1.8654
2025-12-30   0.3532  87237.13  2937.90    78.32   123.28   1.8508
2025-12-31   0.3514  88485.49  2973.69    78.89   125.01   1.8776


In [3]:
# Descriptive statistics per asset
print('Asset Statistics (annualized):')
stats = pd.DataFrame({
    'AnnReturn': crypto_returns.mean() * 252,
    'AnnVol': crypto_returns.std() * np.sqrt(252),
    'Sharpe': crypto_returns.mean() / crypto_returns.std() * np.sqrt(252),
    'MaxDD': ((1 + crypto_returns).cumprod() / (1 + crypto_returns).cumprod().cummax() - 1).min(),
    'Skew': crypto_returns.skew(),
    'Kurtosis': crypto_returns.kurtosis()
})
print(stats.round(3).to_string())

# Correlation matrix between crypto assets
print('\nAsset Correlation Matrix:')
print(crypto_returns.corr().round(3).to_string())

Asset Statistics (annualized):
         AnnReturn  AnnVol  Sharpe  MaxDD   Skew  Kurtosis
symbol                                                    
ADAUSDT      0.458   0.839   0.546 -0.918  1.814    20.832
BTCUSDT      0.381   0.486   0.783 -0.766  0.148     3.661
ETHUSDT      0.481   0.660   0.730 -0.793  0.192     4.686
LTCUSDT      0.320   0.734   0.436 -0.888 -0.070     5.863
SOLUSDT      0.955   0.994   0.960 -0.963  0.590     6.277
XRPUSDT      0.624   0.901   0.693 -0.832  2.520    28.375

Asset Correlation Matrix:
symbol   ADAUSDT  BTCUSDT  ETHUSDT  LTCUSDT  SOLUSDT  XRPUSDT
symbol                                                       
ADAUSDT    1.000    0.646    0.691    0.635    0.559    0.599
BTCUSDT    0.646    1.000    0.798    0.712    0.559    0.541
ETHUSDT    0.691    0.798    1.000    0.738    0.628    0.566
LTCUSDT    0.635    0.712    0.738    1.000    0.513    0.593
SOLUSDT    0.559    0.559    0.628    0.513    1.000    0.462
XRPUSDT    0.599    0.541    0.566  

In [4]:
## Crypto Strategy Signals

def sma_crossover_signal(prices, fast=50, slow=200):
    fast_ema = prices.rolling(fast).mean()
    slow_ema = prices.rolling(slow).mean()
    return (fast_ema > slow_ema).astype(float)

def momentum_signal(prices, lookback=63):
    return (prices / prices.shift(lookback) > 1.0).astype(float)

def trend_weather_signal(prices, vol_window=63):
    trend = prices.rolling(200).mean()
    vol = prices.pct_change().rolling(vol_window).std() * np.sqrt(252)
    vol_median = vol.rolling(252).median()
    long = (prices > trend) & (vol < vol_median * 1.5)
    return long.astype(float)

def har_vol_target_signal(prices, rv_window=22):
    realized_var = prices.pct_change().rolling(rv_window).var() * 252
    target_vol = 0.15
    scaling = np.clip(target_vol / np.sqrt(realized_var + 1e-8), 0.2, 2.0)
    return scaling

# Identify BTC column
btc_col = [c for c in crypto_returns.columns if 'BTC' in str(c)]
strategies = {}

if btc_col:
    btc = crypto_returns[btc_col[0]]
    btc_prices = (1 + btc).cumprod()

    # 1. EMA-Cross-Crypto (BTC SMA20/50 crossover)
    ema_signal = sma_crossover_signal(btc_prices, 20, 50)
    strategies['EMA-Cross-Crypto'] = btc * ema_signal.shift(1)

    # 2. Trend-Follow-Crypto (BTC SMA50/200 with vol filter)
    tf_signal = trend_weather_signal(btc_prices)
    strategies['TrendFollow-Crypto'] = btc * tf_signal.shift(1)

    # 3. Momentum-Crypto (BTC 3-month momentum)
    mom_signal = momentum_signal(btc_prices, 63)
    strategies['Momentum-Crypto'] = btc * mom_signal.shift(1)

    # 4. HAR-RV-J vol-target BTC (M12 strategy)
    vol_scale = har_vol_target_signal(btc_prices)
    strategies['HAR-RV-J-VolTarget'] = btc * vol_scale.shift(1)

    # 5. MultiCanal (equal-weight all crypto assets)
    strategies['MultiCanal'] = crypto_returns.mean(axis=1)

    # 6. ETH-BTC relative momentum (long ETH when outperforming BTC)
    eth_col = [c for c in crypto_returns.columns if 'ETH' in str(c)]
    if eth_col:
        eth = crypto_returns[eth_col[0]]
        eth_prices = (1 + eth).cumprod()
        rel_mom = (eth_prices / btc_prices).rolling(20).mean()
        eth_signal = (eth_prices / eth_prices.shift(20) > btc_prices / btc_prices.shift(20)).astype(float)
        strategies['ETH-RelMomentum'] = eth * eth_signal.shift(1)

strat_df = pd.DataFrame(strategies).dropna()
print(f'Strategies: {len(strat_df.columns)}')
print(f'Observations: {len(strat_df)}')
print()
for name, rets in strat_df.items():
    sharpe = rets.mean() / rets.std() * np.sqrt(252) if rets.std() > 0 else 0
    ann_ret = rets.mean() * 252
    print(f'  {name}: Sharpe={sharpe:.3f}, AnnRet={ann_ret:.4f}')

Strategies: 6
Observations: 1944

  EMA-Cross-Crypto: Sharpe=0.872, AnnRet=0.3092
  TrendFollow-Crypto: Sharpe=0.542, AnnRet=0.1536
  Momentum-Crypto: Sharpe=0.948, AnnRet=0.3427
  HAR-RV-J-VolTarget: Sharpe=0.845, AnnRet=0.1452
  MultiCanal: Sharpe=0.881, AnnRet=0.5529
  ETH-RelMomentum: Sharpe=0.513, AnnRet=0.2383


## Correlation Matrix

In [5]:
corr_matrix = strat_df.corr()
print('Strategy Correlation Matrix:')
print(corr_matrix.round(3).to_string())

# Average inter-strategy correlation
upper_tri = []
for i, j in combinations(range(len(corr_matrix)), 2):
    upper_tri.append(corr_matrix.iloc[i, j])
avg_corr = np.mean(upper_tri)
print(f'\nAverage inter-strategy correlation: {avg_corr:.3f}')

# BTC-dependent vs diversified correlation
btc_strats = ['EMA-Cross-Crypto', 'TrendFollow-Crypto', 'Momentum-Crypto', 'HAR-RV-J-VolTarget']
diversified_strats = ['MultiCanal', 'ETH-RelMomentum']
existing_btc = [s for s in btc_strats if s in corr_matrix.columns]
existing_div = [s for s in diversified_strats if s in corr_matrix.columns]
if existing_btc and len(existing_btc) > 1:
    btc_corr = corr_matrix.loc[existing_btc, existing_btc].values
    btc_avg = btc_corr[np.triu_indices(len(existing_btc), k=1)].mean()
    print(f'BTC-strategies avg correlation: {btc_avg:.3f}')
if existing_btc and existing_div:
    cross = corr_matrix.loc[existing_btc, existing_div].values.mean()
    print(f'BTC vs diversified correlation: {cross:.3f}')

Strategy Correlation Matrix:
                    EMA-Cross-Crypto  TrendFollow-Crypto  Momentum-Crypto  HAR-RV-J-VolTarget  MultiCanal  ETH-RelMomentum
EMA-Cross-Crypto               1.000               0.537            0.819               0.696       0.568            0.437
TrendFollow-Crypto             0.537               1.000            0.631               0.657       0.488            0.297
Momentum-Crypto                0.819               0.631            1.000               0.706       0.563            0.419
HAR-RV-J-VolTarget             0.696               0.657            0.706               1.000       0.781            0.492
MultiCanal                     0.568               0.488            0.563               0.781       1.000            0.596
ETH-RelMomentum                0.437               0.297            0.419               0.492       0.596            1.000

Average inter-strategy correlation: 0.579
BTC-strategies avg correlation: 0.674
BTC vs diversified correlatio

## Portfolio Blending Analysis

In [6]:
# Crypto sleeve weights (within Binance sleeve)
crypto_weights = {
    'EMA-Cross-Crypto': 0.40,
    'TrendFollow-Crypto': 0.15,
    'Momentum-Crypto': 0.10,
    'HAR-RV-J-VolTarget': 0.20,
    'MultiCanal': 0.10,
    'ETH-RelMomentum': 0.05,
}

# Sweep allocation weights (what fraction goes to each strategy)
print('Crypto Sleeve Blending Results:')
print(f'{"Strategy":>22} {"Weight":>8} {"Sharpe":>8} {"AnnRet":>8} {"AnnVol":>8} {"MaxDD":>8}')
print('-' * 70)

for name, w in crypto_weights.items():
    if name in strat_df.columns:
        rets = strat_df[name]
        sharpe = rets.mean() / rets.std() * np.sqrt(252) if rets.std() > 0 else 0
        ann_ret = rets.mean() * 252
        ann_vol = rets.std() * np.sqrt(252)
        cum = (1 + rets).cumprod()
        max_dd = ((cum / cum.cummax()) - 1).min()
        print(f'{name:>22} {w:>8.0%} {sharpe:>8.3f} {ann_ret:>8.4f} {ann_vol:>8.4f} {max_dd:>8.3f}')

# Blended portfolio
blend_rets = pd.Series(0.0, index=strat_df.index)
for name, w in crypto_weights.items():
    if name in strat_df.columns:
        blend_rets += strat_df[name] * w

sharpe = blend_rets.mean() / blend_rets.std() * np.sqrt(252)
ann_ret = blend_rets.mean() * 252
ann_vol = blend_rets.std() * np.sqrt(252)
cum = (1 + blend_rets).cumprod()
max_dd = ((cum / cum.cummax()) - 1).min()
print('-' * 70)
print(f'{"BLENDED PORTFOLIO":>22} {"100%":>8} {sharpe:>8.3f} {ann_ret:>8.4f} {ann_vol:>8.4f} {max_dd:>8.3f}')

# Compare to BTC buy-and-hold
btc_col_name = [c for c in strat_df.columns if 'BTC' in str(c) or 'EMA' in str(c)]
if btc_col and btc_col[0] in crypto_returns.columns:
    btc_bh = crypto_returns[btc_col[0]].loc[strat_df.index]
    bh_sharpe = btc_bh.mean() / btc_bh.std() * np.sqrt(252)
    bh_ret = btc_bh.mean() * 252
    bh_vol = btc_bh.std() * np.sqrt(252)
    bh_cum = (1 + btc_bh).cumprod()
    bh_dd = ((bh_cum / bh_cum.cummax()) - 1).min()
    print(f'{"BTC Buy-and-Hold":>22} {"100%":>8} {bh_sharpe:>8.3f} {bh_ret:>8.4f} {bh_vol:>8.4f} {bh_dd:>8.3f}')
    print(f'\nDelta Sharpe (blended vs BTC B&H): {sharpe - bh_sharpe:+.3f}')

Crypto Sleeve Blending Results:
              Strategy   Weight   Sharpe   AnnRet   AnnVol    MaxDD
----------------------------------------------------------------------
      EMA-Cross-Crypto      40%    0.872   0.3092   0.3546   -0.579


    TrendFollow-Crypto      15%    0.542   0.1536   0.2834   -0.358
       Momentum-Crypto      10%    0.948   0.3427   0.3614   -0.507
    HAR-RV-J-VolTarget      20%    0.845   0.1452   0.1718   -0.372
            MultiCanal      10%    0.881   0.5529   0.6274   -0.820
       ETH-RelMomentum       5%    0.513   0.2383   0.4642   -0.709
----------------------------------------------------------------------
     BLENDED PORTFOLIO     100%    0.971   0.2773   0.2854   -0.517
      BTC Buy-and-Hold     100%    0.820   0.3989   0.4862   -0.766

Delta Sharpe (blended vs BTC B&H): +0.151


## Recommendations

Based on the correlation analysis and blended Sharpe, the optimal allocation will be reported here.

In [7]:
print('=' * 60)
print('PORTFOLIO HYBRIDE — PHASE 1 CRYPTO SLEEVE SUMMARY')
print('=' * 60)
print(f'Period: {strat_df.index[0].date()} to {strat_df.index[-1].date()}')
print(f'Days: {len(strat_df)}')
print(f'Strategies modeled: {len(strat_df.columns)}')
print(f'Average inter-strategy correlation: {avg_corr:.3f}')
print()
print('Crypto sleeve:')
blend_sharpe = blend_rets.mean() / blend_rets.std() * np.sqrt(252)
print(f'  Blended Sharpe: {blend_sharpe:.3f}')
print(f'  Blended AnnRet: {blend_rets.mean()*252:.4f}')
print(f'  Blended AnnVol: {blend_rets.std()*np.sqrt(252):.4f}')
print()
print('Phase 2 next steps:')
print('  1. Equity sleeve via QC Cloud Playwright (Security Master required)')
print('  2. Cross-sleeve correlation analysis (equity vs crypto)')
print('  3. Full backtest unified 2018-2025')
print('  4. Walk-forward annual validation')
print()
print('Phase 3-5:')
print('  Walk-forward + multi-seed -> Paper trading 30j -> Live nodes')

PORTFOLIO HYBRIDE — PHASE 1 CRYPTO SLEEVE SUMMARY
Period: 2020-09-05 to 2025-12-31
Days: 1944
Strategies modeled: 6
Average inter-strategy correlation: 0.579

Crypto sleeve:
  Blended Sharpe: 0.971
  Blended AnnRet: 0.2773
  Blended AnnVol: 0.2854

Phase 2 next steps:
  1. Equity sleeve via QC Cloud Playwright (Security Master required)
  2. Cross-sleeve correlation analysis (equity vs crypto)
  3. Full backtest unified 2018-2025
  4. Walk-forward annual validation

Phase 3-5:
  Walk-forward + multi-seed -> Paper trading 30j -> Live nodes
